In [6]:
!pip install git+https://github.com/openai/CLIP.git
!pip install torch torchvision Pillow

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-gatnnuby
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-gatnnuby
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=9cb35b0c0d11b4274ca8a7195de1ee0e68455d7b631bcd7e6b32d6248d8cc1f3
  Stored in directory: /tmp/pip-ephem-wheel-cache-8yjs6_5l/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [7]:
import clip
import torch
from PIL import Image
from IPython.display import display
import numpy as np

In [8]:
# available models = ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']

model, preprocess = clip.load("ViT-B/32")

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 110MiB/s]


In [10]:
import requests, os
from PIL import Image
from io import BytesIO

images_data = [
    ("sunset_mountains", "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400"),
    ("dog_playing", "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=400"),
    ("city_night", "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?w=400"),
    ("pasta_bowl", "https://images.unsplash.com/photo-1555949258-eb67b1ef0ceb?w=400"),
    ("person_reading", "https://images.unsplash.com/photo-1524995997946-a1c2e315a42f?w=400"),
    ("snowy_forest", "https://images.unsplash.com/photo-1418985991508-e47386d96a71?w=400"),
    ("cat_window", "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400"),
    ("beach_water", "https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=400"),
    ("red_sports_car", "https://images.unsplash.com/photo-1544636331-e26879cd4d9b?w=400"),
    ("coffee_table", "https://images.unsplash.com/photo-1495474472287-4d71bcdd2085?w=400"),
]

os.makedirs("test_images", exist_ok=True)

for name, url in images_data:
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    img.save(f"test_images/{name}.jpg")
    print(f"Saved {name}.jpg")

Saved sunset_mountains.jpg
Saved dog_playing.jpg
Saved city_night.jpg
Saved pasta_bowl.jpg
Saved person_reading.jpg
Saved snowy_forest.jpg
Saved cat_window.jpg
Saved beach_water.jpg
Saved red_sports_car.jpg
Saved coffee_table.jpg


In [11]:
image = Image.open("/content/test_images/beach_water.jpg")

#display(preprocess(image))
image = preprocess(image).unsqueeze(0)

In [12]:
text = ["sea view","water","ice"]
text = clip.tokenize(text)

In [13]:
with torch.no_grad():
  logits,_ = model(image,text)

In [14]:
logits.softmax(dim=1)

tensor([[0.7721, 0.1803, 0.0476]])

In [15]:
test_queries = [
    "sunset over mountains",
    "a dog playing in grass",
    "busy city street at night",
    "a bowl of pasta",
    "person reading a book",
    "snow covered forest",
    "cat sitting on a windowsill",
    "beach with clear blue water",
    "a red sports car",
    "coffee cup on a wooden table",
    # Cross-queries to test CLIP's understanding:
    "warm drink in a cafe",        # should match coffee
    "animal near window",          # should match cat
    "fast automobile",             # should match sports car
    "Italian food",                # should match pasta
]

In [16]:
def load_img(img):
  img = Image.open(img)
  img = preprocess(img).unsqueeze(0)
  return img


In [17]:
path = "/content/test_images/"
imgs = [load_img(path+x) for x in os.listdir(path)]

In [18]:
texts = clip.tokenize(test_queries)

In [19]:
texts

tensor([[49406,  3424,   962,  ...,     0,     0,     0],
        [49406,   320,  1929,  ...,     0,     0,     0],
        [49406,  4354,  1305,  ...,     0,     0,     0],
        ...,
        [49406,  4668,  2252,  ...,     0,     0,     0],
        [49406,  1953, 25258,  ...,     0,     0,     0],
        [49406,  5175,  1559,  ...,     0,     0,     0]], dtype=torch.int32)

In [20]:
with torch.no_grad():
  logits, _ = model(imgs[0], texts)

In [21]:
def prediction(img):
  with torch.no_grad():
    logits, _ = model(img, texts)
    probs = logits.softmax(dim=-1)[0]
  mapping = {text:prob for text,prob in zip( test_queries, probs)}

  return max(mapping,key=mapping.get)

probs = prediction(imgs[0])
probs

'busy city street at night'

In [22]:
predictions = [prediction(img) for img in imgs]

In [23]:
predictions

['busy city street at night',
 'snow covered forest',
 'sunset over mountains',
 'cat sitting on a windowsill',
 'a dog playing in grass',
 'beach with clear blue water',
 'a bowl of pasta',
 'fast automobile',
 'warm drink in a cafe',
 'person reading a book']

# Search

In [24]:
imgs = [Image.open(path + x) for x in os.listdir(path)]
imgs

[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x246>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x267>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x267>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x275>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x267>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x266>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x267>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x300>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x267>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=400x267>]

In [25]:
def encode_images(imgs, model, preprocess):
  tensors = torch.stack([preprocess(img) for img in imgs])

  with torch.no_grad():
    embeddings = model.encode_image(tensors)
    embeddings /= embeddings.norm(dim=-1,keepdim=True)
  return embeddings

img_embeddings = encode_images(imgs,model,preprocess)

In [26]:
def encode_text(queries, model):
  tokens = clip.tokenize(queries)

  with torch.no_grad():
    embeddings = model.encode_text(tokens)
    embeddings /= embeddings.norm(dim=-1, keepdim=True)
    return embeddings

txt_embeddings = encode_text(test_queries, model)

In [32]:
def search(query, image_embeddings, model):
  text_emb = encode_text(query,model)

  similarities = (image_embeddings @ text_emb.T).squeeze()
  top_ids = similarities.topk(3)

  paths = [(path + x) for x in os.listdir(path)]

  results = []
  for i in top_ids.indices:
    results.append({
        "image":paths[i],
        "score":similarities[i].item()
    })
  return results

top_results = search("Sea View",img_embeddings,model)

In [34]:
top_results

[{'image': '/content/test_images/beach_water.jpg',
  'score': 0.2431623488664627},
 {'image': '/content/test_images/city_night.jpg', 'score': 0.2137220948934555},
 {'image': '/content/test_images/dog_playing.jpg',
  'score': 0.20924869179725647}]